In [1]:
import threading
import queue

from arraylake import Client
import icechunk
import datetime
import numpy as np
import zarr

from anemoi.inference.runners.simple import SimpleRunner
from anemoi.inference.outputs.printer import print_state
import torch

from main import fetch_initial_conditions, get_gpu_regridder, state_to_xarray, datetime_to_str

In [2]:
client = Client()
client.login()

🔓 Successfully refreshed tokens!

> Token stored at /home/mambauser/.arraylake/token.json

╭──────────────────────────────────────────────── 👤 User Details ────────────────────────────────────────────────╮
│ Name: Joe Hamman                                                                                                │
│ Email: joe@earthmover.io                                                                                        │
│ Id: 33d94735-8896-4a91-bd1f-d595f39b39d0                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🔓 Successfully logged in!

> Token stored at /home/mambauser/.arraylake/token.json

╭──────────────────────────────────────────────── 👤 User Details ────────────────────────────────────────────────╮
│ Name: Joe Hamman                                                                                                │
│ Email: joe@earthmover.io                                                                                        │
│ Id: 33d94735-8896-4a91-bd1f-d595f39b39d0                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [3]:
repo = client.get_repo("earthmover-public/aifs-initial-conditions")
session = repo.readonly_session("main")
session

In [18]:
date = datetime.datetime(2025, 9, 15, 6, 0, 0, tzinfo=datetime.UTC)
print("loading initial conditions for", date)
%time fields = fetch_initial_conditions(date, session)

loading initial conditions for 2025-09-15 06:00:00+00:00
CPU times: user 980 ms, sys: 645 ms, total: 1.63 s
Wall time: 1.09 s


In [19]:
print("setting up regridder")
regridder = get_gpu_regridder({"grid": "N320"}, {"grid": (0.25, 0.25)})

setting up regridder


In [20]:
checkpoint = {"huggingface": "ecmwf/aifs-single-1.0"}
runner = SimpleRunner(checkpoint, device="cuda")

In [21]:
target_repo = client.get_or_create_repo("earthmover-public/aifs-outputs")
target_session = target_repo.writable_session("main")

  2025-09-15T15:48:33.172937Z  WARN icechunk::asset_manager: A snapshot with 8662 nodes is being loaded into the cache that can only keep 30000 nodes. Consider increasing the size of the snapshot cache using the num_snapshot_nodes field in CachingConfig
    at icechunk/src/asset_manager.rs:251



In [22]:
date_no_tz = date.replace(tzinfo=None)
input_state = dict(date=date_no_tz, fields=fields)

# we put data that we want to write into a queue
q = queue.Queue()
lock = threading.Lock()

def worker():
    while True:
        (ds, store, group_name, kwargs) = q.get()
        # lock is probably unncessary
        with lock:
            ds.to_zarr(
                store, group=group_name, zarr_format=3, consolidated=False, **kwargs
            )
        q.task_done()

# a separate thread for I/O to avoid blocking the main loop
threading.Thread(target=worker, daemon=True).start()

print("starting forecast loop")
kwargs = {"mode": "w"}
# main forecast loop

# clear GPU memory
torch.cuda.empty_cache()

for n, state in enumerate(runner.run(input_state=input_state, lead_time=96)):
    print_state(state)
    ds = state_to_xarray(state, regridder=regridder).chunk()
    group = datetime_to_str(date)
    if n > 0:
        kwargs = {"mode": "a", "append_dim": "valid_time"}
    q.put((ds, target_session.store, group, kwargs))

q.join()  # wait for all I/O tasks to finish

# clear GPU memory
torch.cuda.empty_cache()


starting forecast loop


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

/opt/coiled/env/lib/python3.12/site-packages/anemoi/utils/config.py:209: UserWarning: Modifying an instance of DotDict(). This class is intended to be immutable.
  warnings.warn("Modifying an instance of DotDict(). This class is intended to be immutable.")



😀 date=2025-09-15T12:00:00 latitudes=(542080,) longitudes=(542080,) fields=102

    q_50   shape=(542080,) min=9.45981e-07    max=3.35365e-06   
    t_1000 shape=(542080,) min=234.707        max=318.724       
    v_925  shape=(542080,) min=-31.4245       max=31.8512       
    z_850  shape=(542080,) min=8496.78        max=15934.3       
    swvl2  shape=(542080,) min=0              max=0.759633      
    tcc    shape=(542080,) min=0              max=1             


😀 date=2025-09-15T18:00:00 latitudes=(542080,) longitudes=(542080,) fields=102

    q_50   shape=(542080,) min=9.49654e-07    max=3.30761e-06   
    t_1000 shape=(542080,) min=234.324        max=316.646       
    v_925  shape=(542080,) min=-32.0755       max=34.0836       
    z_850  shape=(542080,) min=8476.11        max=15781.6       
    swvl2  shape=(542080,) min=0              max=0.755325      
    tcc    shape=(542080,) min=0              max=1             


😀 date=2025-09-16T00:00:00 latitudes=(542080,) longitud

In [23]:
target_session.commit("Wrote a 96 hour forecast during demo")

'Z6SKK0VZV25V11C2J50G'